# 10 · Explainable Anomaly Detection

## What this notebook covers
Flagging an anomaly without explanation is often insufficient for production use. A fraud alert, a manufacturing defect flag, or a security incident detection all require: *why* is this anomalous? This notebook implements SHAP-based explanation of anomaly scores and contrastive explanations.

## The explainability challenge for anomaly detection
Standard SHAP is designed for supervised models with a labelled target. Anomaly detectors have a continuous *score* but no natural class labels. We treat the anomaly score as the target and apply SHAP to explain *what drives the score up or down for a specific point*.

A **contrastive explanation** answers: "compared to the nearest normal point, what features differ most?" This is often more interpretable for domain experts than global SHAP values.

In [ ]:
import numpy as np
import pandas as pd
import shap
from sklearn.ensemble import IsolationForest
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

# Dataset: 8 named features
feature_names = ["cpu_pct", "mem_pct", "disk_io", "net_in", "net_out", "error_rate", "latency_ms", "req_per_sec"]
n = 500
X_normal = np.column_stack([
    np.random.normal(50, 10, n),   # cpu
    np.random.normal(60, 8, n),    # mem
    np.random.normal(30, 5, n),    # disk
    np.random.normal(100, 20, n),  # net_in
    np.random.normal(80, 15, n),   # net_out
    np.random.normal(0.01, 0.005, n),  # error_rate
    np.random.normal(200, 30, n),  # latency
    np.random.normal(1000, 100, n) # req/s
])
X_anom = np.array([
    [95, 62, 31, 105, 82, 0.012, 850, 200],  # high cpu + high latency + low req
    [52, 95, 32, 108, 79, 0.009, 210, 990],  # high memory
    [51, 61, 29, 102, 83, 0.08,  220, 980],  # high error rate
])
X_all = np.vstack([X_normal, X_anom])
y_all = np.array([0]*n + [1]*3)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)
X_train_scaled = X_scaled[:n]
print(f"Dataset: {len(X_all)} points  |  3 injected anomalies")

In [ ]:
iso = IsolationForest(n_estimators=200, contamination=0.01, random_state=0)
iso.fit(X_train_scaled)
scores = -iso.score_samples(X_scaled)

anomaly_indices = np.where(y_all == 1)[0]
for idx in anomaly_indices:
    print(f"Point {idx}: score={scores[idx]:.4f}  |  raw features: {dict(zip(feature_names, X_all[idx].round(2)))}")

In [ ]:
# ── SHAP on anomaly scores ─────────────────────────────────────────────────────
# Use a TreeExplainer on IsolationForest
# Note: shap.TreeExplainer supports IsolationForest natively
explainer = shap.TreeExplainer(iso)
shap_values = explainer.shap_values(X_scaled)

# Per-anomaly explanation
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, idx in zip(axes, anomaly_indices):
    sv = shap_values[idx]
    sorted_idx = np.argsort(np.abs(sv))[::-1]
    colours = ["tomato" if sv[i] > 0 else "steelblue" for i in sorted_idx]
    ax.barh([feature_names[i] for i in sorted_idx], sv[sorted_idx], color=colours)
    ax.set_title(f"Point {idx}  score={scores[idx]:.3f}")
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("SHAP value (positive = more anomalous)")

plt.suptitle("SHAP explanations for anomalous points", fontsize=12)
plt.tight_layout()
plt.savefig(f"{base}/10_shap_anomaly.png", dpi=120)
plt.show()

In [ ]:
# ── Contrastive explanation ────────────────────────────────────────────────────
def contrastive_explain(query_idx, X_all, X_scaled, y_all, scores, feature_names, k=5):
    """
    Compare query point to its k nearest normal neighbours.
    Returns a DataFrame of feature deltas sorted by absolute difference.
    """
    nn = NearestNeighbors(n_neighbors=k+1, metric="euclidean")
    nn.fit(X_scaled[y_all == 0])
    dists, idxs = nn.kneighbors([X_scaled[query_idx]])
    # idxs are indices into the normal subset
    normal_subset = X_all[y_all == 0]
    reference = normal_subset[idxs[0][1:k+1]].mean(axis=0)
    query     = X_all[query_idx]
    delta     = query - reference
    df = pd.DataFrame({
        "feature"  : feature_names,
        "query"    : query.round(3),
        "reference_mean": reference.round(3),
        "delta"    : delta.round(3),
        "|delta|"  : np.abs(delta).round(3),
    }).sort_values("|delta|", ascending=False)
    return df

for idx in anomaly_indices:
    print(f"\n=== Contrastive explanation for anomaly at index {idx} ===")
    exp = contrastive_explain(idx, X_all, X_scaled, y_all, scores, feature_names)
    print(exp.head(4).to_string(index=False))